# Fine-Tuning Laya (multilingual) on the Nepali typed-decision dataset

Adapted from the upstream `laya_finetune_typed_decisions_2xT4_kaggle.ipynb` for
`convaiinnovations/laya` -> **multilingual** subfolder and the `ne-decisions-v1`
export (Devanagari restaurant commands; 5 typed questions per case).

### Kaggle settings
* Accelerator: **GPU T4 x2**
* Internet: **On**
* Data: upload `data/export/ne-decisions-v1-{train,calibration}.jsonl`,
  `data/benchmark/cl-bench-v1.json` and `data/benchmark/ne-probe-v1.json`
  as one private Kaggle dataset and mount it at `/kaggle/input/laya-nepali-clothing-v1`.

`TRAIN_LIMIT` below slices the training file for a scaling run (0 = all rows).


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()} | visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
assert n_gpu >= 2, "Select GPU T4 x2 in Notebook options."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install -q -U "laya>=0.3.20" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)


## 1. Model, tokenizer and data

In [ ]:
import glob, json, os

import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from laya.agent import _fix_tokenizer_config
from laya.common import QTYPES, build_sequence, render_options

MODEL_ID = "convaiinnovations/laya"
SUBFOLDER = "multilingual"          # Nepali routes here
DATA_DIR = "/kaggle/input/laya-nepali-clothing-v1"
found = glob.glob("/kaggle/input/**/cl-decisions-v1-train.jsonl", recursive=True)
if found:
    DATA_DIR = os.path.dirname(found[0])
print("Resolved DATA_DIR:", DATA_DIR)
OUTPUT_DIR = "/kaggle/working/laya_clothing_v1"
TRAIN_LIMIT = 0                     # 0 = all rows; set 400/900/1600 for scaling runs

snapshot = snapshot_download(MODEL_ID)
model_dir = os.path.join(snapshot, SUBFOLDER)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
print("snapshot:", snapshot)
print("config  :", {k: cfg.get(k) for k in ("max_len", "head_max_len", "temperature", "model_name")})

def read_rows(pattern):
    paths = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    assert paths, f"no data for {pattern} under {DATA_DIR}"
    rows = []
    for path in paths:
        rows.extend(json.loads(line) for line in open(path, encoding="utf-8") if line.strip())
    return rows

train_rows = read_rows("cl-decisions-v1-train.jsonl")
calib_rows = read_rows("cl-decisions-v1-calibration.jsonl")
if TRAIN_LIMIT:
    train_rows = train_rows[:TRAIN_LIMIT]
print(f"train rows {len(train_rows)} | calibration rows {len(calib_rows)}")


def build_training_item(state, q, gold_q, qid):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    total = sum(target)
    target = [v / total for v in target] if total > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(
        tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"]
    )
    if len(markers) != k:
        return None
    none_index = list(crit.keys()).index("none") if t == "choice" and "none" in crit else -1
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label,
            "none_index": none_index, "qid": qid}


SAFETY_FAMILIES = {"skin_sensitivity", "counterfeit_concern", "sizing_dispute", "refund_demand", "complaint", "human", "bare_order", "yes_no", "gibberish", "unclear"}
SAFETY_LOSS_WEIGHT = 5.0


def build_items(rows):
    items = []
    for row in rows:
        state = json.loads(row["state"])
        questions = json.loads(row["questions"])
        gold = json.loads(row["gold"])
        family = str((row.get("provenance") or {}).get("family", ""))
        for qid, q in questions.items():
            if qid in gold:
                item = build_training_item(state, q, gold[qid], qid)
                if item:
                    item["family"] = family
                    item["weight"] = SAFETY_LOSS_WEIGHT if family in SAFETY_FAMILIES else 1.0
                    items.append(item)
    return items


train_items = build_items(train_rows)
calib_items = build_items(calib_rows)
print(f"train items {len(train_items)} | calibration items {len(calib_items)}")
torch.save(train_items, "/kaggle/working/train_items.pt")
torch.save(calib_items, "/kaggle/working/calib_items.pt")


## 2. DDP training script (RLCD, 2xT4)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import answer_confidence, build_model, proper_reward, QTYPES


def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
        "weight": torch.tensor([float(it.get("weight", 1.0)) for it in items]),
    }


def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, : len(z)] = torch.tensor(z)
        T[i, : len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    train_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    calib_items = torch.load("/kaggle/working/calib_items.pt", weights_only=False)
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 4
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": LR_ENCODER}, {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01
    )
    total_updates = max(1, (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"train items {len(train_items)} | calibration {len(calib_items)} | per rank {len(my_items)}")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx : b_idx + MICRO_BATCH]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            weights = batch["weight"].to(device)
            weight_sum = weights.sum().clamp(min=1e-6)
            loss_rl = -(((adv * logp).mean(0)) * weights).sum() / weight_sum
            loss_ce = -(((target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1)) * weights).sum() / weight_sum
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if rank == 0 and (n_batches % 50) == 0:
                print(f"  epoch {epoch + 1}/{EPOCHS} step {n_batches} loss {loss.item() * GRAD_ACCUM:.4f} reward {r.mean().item():.3f}")

        if rank == 0:
            print(f"=== epoch {epoch + 1}/{EPOCHS} done in {time.time() - t0:.1f}s | avg loss {epoch_loss / max(1, n_batches):.4f} ===")
        dist.barrier()

        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            print(f"  saved rolling checkpoint (epoch {epoch + 1})")

    dist.barrier()

    if rank == 0:
        print("\nfitting calibration temperatures on the held-out split...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx : c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device),
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    calib_preds.append((it["qtype"], l_np[r, : len(it["markers"])], it["target"], it))

        fitted = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t, _ in calib_preds if q_type == qt]
                if sel:
                    fitted[qt] = fit_one_temp(sel)
            print("fitted temperatures (choice, score, noul):", [round(t, 3) for t in fitted])
        except Exception as exc:
            print("temperature fitting fallback:", exc)

        SAFETY_FAMILIES = {"skin_sensitivity", "counterfeit_concern", "sizing_dispute", "refund_demand", "complaint", "human", "bare_order", "yes_no", "gibberish", "unclear"}

        command_preds = []
        for q_type, logits, target, item in calib_preds:
            if item.get("qid") != "command":
                continue
            scale = fitted[0] if fitted[0] > 1e-6 else 1.0
            z = logits / scale
            p = np.exp(z - z.max())
            p = p / p.sum()
            none_index = item.get("none_index", -1)
            p_none = float(p[none_index]) if none_index >= 0 else 0.0
            command_preds.append((int(p.argmax()), float(answer_confidence(p, len(logits))), p_none, item))

        tau_grid = (0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9)
        p_grid = (0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 1.0)
        grid = []
        for tau in tau_grid:
            for p_max in p_grid:
                safety_fp = kind_hits = kind_total = 0
                for label, conf, p_none, item in command_preds:
                    fires = conf >= tau and label != item.get("none_index", -1) and p_none < p_max
                    if item.get("family", "") in SAFETY_FAMILIES:
                        safety_fp += int(fires)
                    else:
                        kind_total += 1
                        kind_hits += int(fires and label == int(np.argmax(item["target"])))
                grid.append((tau, p_max, safety_fp, kind_hits, kind_total))
        zero_fp = [row for row in grid if row[2] == 0]
        if zero_fp:
            best_tau, best_p_none, _, best_hits, best_total = max(
                zero_fp, key=lambda row: (row[3], row[0], -row[1])
            )
        else:
            floor = min(row[2] for row in grid)
            best_tau, best_p_none, _, best_hits, best_total = max(
                [row for row in grid if row[2] == floor], key=lambda row: (row[3], row[0], -row[1])
            )
        print(f"rule grid: {len(zero_fp)} zero-FP points of {len(grid)}")
        for tau in tau_grid:
            best_row = max((row for row in zero_fp if row[0] == tau), key=lambda row: row[3], default=None)
            if best_row:
                print(f"  tau {tau:.2f} | best zero-FP p_none<{best_row[1]:.2f} | kind hits {best_row[3]}/{best_row[4]}")
        print(f"chosen rule: tau {best_tau} p_none<{best_p_none} | safety fp {0 if zero_fp else floor} | kind hits {best_hits}/{best_total}")

        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-nepali-clothing-v1"
        cfg["temperature"] = fitted
        cfg["abstain_threshold"] = best_tau
        cfg["abstain_p_none"] = best_p_none
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print("model saved to", output_dir)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_clothing_v1"
cmd = f"torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {OUTPUT_DIR}"
print("executing:", cmd)
!{cmd}


## 3. Evaluate on the frozen Devanagari benchmark (ne-bench-deva-v2)

In [ ]:
import time, json, os
import numpy as np
import laya
from laya.common import ece_score

CLOTHING_CRITERIA = {
    "greet": "hello or greeting", "thanks": "thanks", "goodbye": "goodbye",
    "catalog": "see products or catalog", "recommend": "recommendation",
    "price": "asking price of an item", "size": "asking about available sizes",
    "color": "asking about available colors", "material": "asking fabric or material",
    "stock": "asking whether an item is in stock", "delivery": "delivery area or fee",
    "cod": "asking about cash on delivery", "order_status": "status of an existing order",
    "cart": "see current cart", "checkout": "place order or buy",
    "exchange": "exchange or return item", "custom_order": "custom stitching or tailoring",
    "saved_address": "use saved address", "none": "none of these",
}
CLOTHING_LABEL_TO_KIND = {
    "greet": "social.greet", "thanks": "social.thanks", "goodbye": "social.goodbye",
    "catalog": "discovery.show_catalog", "recommend": "discovery.recommend",
    "price": "discovery.query", "size": "discovery.query", "color": "discovery.query",
    "material": "discovery.query", "stock": "discovery.query",
    "delivery": "fulfillment.ask_delivery", "cod": "fulfillment.ask_cod",
    "order_status": "fulfillment.order_status", "cart": "ordering.view_cart",
    "checkout": "ordering.request_checkout", "exchange": "ordering.request_exchange",
    "custom_order": "ordering.custom_order", "saved_address": "ordering.use_saved_address",
}
CLOTHING_INSTRUCTIONS = (
    "Classify the customer's message for a clothing store. Only choose a command when the "
    "message clearly expresses it. Choose none for: bare item requests like 'yo chaiyo', plain "
    "yes/no, requests for a human or staff, skin sensitivity or allergy questions, refund demands, "
    "counterfeit concerns, and unclear text. A question like 'kati ho?' is a price question, not an order. "
    "order_status means tracking an existing order; exchange means swapping a received item."
)

bench = json.load(open(os.path.join(DATA_DIR, "cl-bench-v1.json"), encoding="utf-8"))
businesses = bench["businesses"]
steps = [step for case in bench["cases"] for step in case["steps"]]
step_meta = {step["id"]: (step["expected"], case["menu_id"]) for case in bench["cases"] for step in case["steps"]}

cfg = json.load(open(os.path.join(OUTPUT_DIR, "rl_agent_config.json"), encoding="utf-8"))
ABSTAIN_THRESHOLD = float(cfg.get("abstain_threshold", 0.5))
ABSTAIN_P_NONE = float(cfg.get("abstain_p_none", 1.0))
print("abstain threshold:", ABSTAIN_THRESHOLD, "| p_none max:", ABSTAIN_P_NONE)
agent = laya.Agent(OUTPUT_DIR, device="cuda")
correct = abstain_total = abstain_correct = 0
safety_false = 0
confs, hits, latencies = [], [], []
safety_failures = []
for step in steps:
    expected_kind = step["expected"]
    case_menu = step_meta[step["id"]][1]
    biz_name = businesses[case_menu]["name"]
    question = {
        "command": {
            "type": "choice",
            "instructions": f"Classify the customer's message for {biz_name}. " + CLOTHING_INSTRUCTIONS,
            "criteria": CLOTHING_CRITERIA,
        }
    }
    t0 = time.perf_counter()
    result = agent.system_one({"customer_message": step["text"]}, question)
    latencies.append((time.perf_counter() - t0) * 1000)
    answer = result["answers"]["command"]
    label = answer.get("choice")
    confidence = float(answer.get("answer_confidence", answer.get("confidence", 0.0)))
    p_none = float((answer.get("probabilities") or {}).get("none", 0.0))
    predicted = None if (
        not label or label == "none" or confidence < ABSTAIN_THRESHOLD or p_none >= ABSTAIN_P_NONE
    ) else CLOTHING_LABEL_TO_KIND.get(label)
    hit = predicted == expected_kind
    hits.append(int(hit))
    confs.append(confidence)
    if hit:
        correct += 1
    if expected_kind is None:
        abstain_total += 1
        if predicted is None:
            abstain_correct += 1
        else:
            safety_false += 1
            safety_failures.append((step["text"], predicted, confidence, p_none))

acc = correct / len(steps) * 100
safety_acc = (abstain_correct / abstain_total * 100) if abstain_total else 100
try:
    ece = ece_score(np.array(confs, dtype=np.float32), np.array(hits, dtype=np.int32))
except Exception as exc:
    print(f"Warning: ece_score failed: {exc}")
    ece = 0.0
print(f"=== CLOTHING BENCHMARK EVALUATION (cl-bench-v1, {len(steps)} steps) ===")
print(f"Accuracy:            {correct}/{len(steps)} ({acc:.1f}%)")
print(f"Safety False Commands: {safety_false} (Abstain accuracy: {safety_acc:.1f}%)")
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"Median Latency:      {np.median(latencies):.1f} ms")
if safety_failures:
    print(f"Sample Safety Failures: {safety_failures[:3]}")


## 4. Publish to the Hugging Face Hub (optional)

In [ ]:
from huggingface_hub import HfApi

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
if not token:
    print("HF_TOKEN secret not set; skipping publish.")
else:


    NEW_REPO = "DontHash/laya-nepali-clothing-v1"
    card = f'''---
    license: apache-2.0
    library_name: transformers
    base_model: convaiinnovations/laya
    language: [ne]
    tags: [laya, nepali, system-one, typed-decisions, calibrated-decisions]
    ---

    # Laya Nepali v1

    Fine-tuned from `convaiinnovations/laya` (multilingual subfolder) on
    `ne-decisions-v1` (Devanagari restaurant commands, 5 typed questions per case)
    with RLCD and per-type temperature fitting.

    Frozen benchmark `ne-bench-deva-v2`: {correct}/{len(steps)} ({correct / len(steps) * 100:.1f}%),
    abstain {abstain_correct}/{abstain_total}, safety false commands {safety_false}.

    Training data is synthetic and reviewed; see the dataset card for provenance.
    '''
    open(os.path.join(OUTPUT_DIR, "README.md"), "w").write(card)
    api = HfApi(token=token)
    api.create_repo(NEW_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=NEW_REPO, repo_type="model")
    print("published to", NEW_REPO)
